# Module 1 — Language Detection

**Part of the RAG-Based E-commerce Customer Support Chatbot**

This notebook trains a lightweight, fast language-identification model that will be used as
**Stage 1** of the chatbot pipeline: every incoming customer message is first checked for its
language before sentiment, intent, and RAG stages run.

**Approach:** TF-IDF (character n-grams, since language ID works better on character-level
features than word-level) + a linear multi-class classifier (`SGDClassifier` with log loss,
i.e. logistic regression trained with SGD — fast to train and fast at inference time).

**Dataset:** [`papluca/language-identification`](https://huggingface.co/datasets/papluca/language-identification)
(20 languages, ~90k train / 10k valid / 10k test rows, already split for us).

**Output artifacts saved to disk (for reuse in the Flask app in Notebook 5):**
- `language_tfidf_vectorizer.joblib`
- `language_classifier.joblib`
- `language_label_encoder.joblib`


## 1. Install dependencies

In [ ]:
!pip install -q datasets scikit-learn joblib pandas


## 2. Imports

In [ ]:
import re
import joblib
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

RANDOM_STATE = 42


## 3. Load the dataset

`papluca/language-identification` ships with `train`, `validation`, and `test` splits already
created, and two columns: `text` and `labels` (an ISO-639-1 language code, e.g. `en`, `ar`, `fr`).


In [ ]:
dataset = load_dataset("papluca/language-identification")
print(dataset)

train_df = dataset["train"].to_pandas()
valid_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

print(train_df.shape, valid_df.shape, test_df.shape)
train_df.head()


README.md:   0%|          | 0.00/4.99k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 12.0MB            

train.csv: downloading bytes:           |  0.00B            

valid.csv:   0%|          | 0.00/1.71M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.69M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/70000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'text'],
        num_rows: 70000
    })
    validation: Dataset({
        features: ['labels', 'text'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['labels', 'text'],
        num_rows: 10000
    })
})
(70000, 2) (10000, 2) (10000, 2)


,labels,text
0,pt,"os chefes de defesa da estónia, letónia, lituâ..."
1,bg,размерът на хоризонталната мрежа може да бъде ...
2,zh,很好，以前从不去评价，不知道浪费了多少积分，现在知道积分可以换钱，就要好好评价了，后来我就把...
3,th,สำหรับ ของเก่า ที่ จริงจัง ลอง honeychurch ...
4,ru,Он увеличил давление .


In [ ]:
print("Languages in the dataset:", sorted(train_df['labels'].unique()))
print("\nClass balance (train):")
print(train_df['labels'].value_counts())


Languages in the dataset: ['ar', 'bg', 'de', 'el', 'en', 'es', 'fr', 'hi', 'it', 'ja', 'nl', 'pl', 'pt', 'ru', 'sw', 'th', 'tr', 'ur', 'vi', 'zh']

Class balance (train):
labels
pt    3500
bg    3500
zh    3500
th    3500
ru    3500
pl    3500
ur    3500
sw    3500
tr    3500
es    3500
ar    3500
it    3500
hi    3500
de    3500
el    3500
nl    3500
fr    3500
vi    3500
en    3500
ja    3500
Name: count, dtype: int64


## 4. Preprocessing

Language ID is intentionally kept *minimal*: we do **not** lowercase aggressively or strip
accents, since case/diacritics/script are themselves strong signals of language. We only
strip leading/trailing whitespace and collapse internal whitespace, and drop empty rows.


In [ ]:
def basic_clean(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\s+", " ", text).strip()
    return text

for df in (train_df, valid_df, test_df):
    df["text"] = df["text"].apply(basic_clean)

train_df = train_df[train_df["text"].str.len() > 0].reset_index(drop=True)
valid_df = valid_df[valid_df["text"].str.len() > 0].reset_index(drop=True)
test_df = test_df[test_df["text"].str.len() > 0].reset_index(drop=True)

print(train_df.shape, valid_df.shape, test_df.shape)


(70000, 2) (10000, 2) (10000, 2)


## 5. Label encoding

We fit a `LabelEncoder` on the language codes so the classifier works with integer targets,
and keep the encoder around so we can map predictions back to language codes at inference time.


In [ ]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df["labels"])
y_valid = label_encoder.transform(valid_df["labels"])
y_test = label_encoder.transform(test_df["labels"])

print("Number of classes:", len(label_encoder.classes_))
print(label_encoder.classes_)


Number of classes: 20
['ar' 'bg' 'de' 'el' 'en' 'es' 'fr' 'hi' 'it' 'ja' 'nl' 'pl' 'pt' 'ru'
 'sw' 'th' 'tr' 'ur' 'vi' 'zh']


## 6. TF-IDF vectorization (character n-grams)

Character n-grams (3–5 chars) are the standard, robust choice for language identification:
they capture spelling/morphology patterns that generalize even for short texts, and they work
uniformly across languages with very different word-segmentation rules (e.g. Arabic vs English).


In [ ]:
vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    max_features=10000,
    sublinear_tf=True,
)

X_train = vectorizer.fit_transform(train_df["text"])
X_valid = vectorizer.transform(valid_df["text"])
X_test = vectorizer.transform(test_df["text"])

print("TF-IDF matrix shape (train):", X_train.shape)


TF-IDF matrix shape (train): (70000, 10000)


## 7. Train the classifier

In [ ]:
clf = SGDClassifier(
    loss="log_loss",       # gives us predict_proba
    alpha=1e-5,
    max_iter=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

clf.fit(X_train, y_train)
print("Training complete.")


Training complete.


## 8. Evaluate

In [ ]:
def evaluate(name, X, y_true):
    y_pred = clf.predict(X)
    acc = accuracy_score(y_true, y_pred)
    print(f"=== {name} accuracy: {acc:.4f} ===")
    print(classification_report(
        y_true, y_pred,
        target_names=label_encoder.classes_,
        zero_division=0,
    ))
    return acc

_ = evaluate("Validation", X_valid, y_valid)
test_acc = evaluate("Test", X_test, y_test)


=== Validation accuracy: 0.9941 ===
              precision    recall  f1-score   support

          ar       1.00      0.99      0.99       500
          bg       1.00      1.00      1.00       500
          de       1.00      1.00      1.00       500
          el       1.00      1.00      1.00       500
          en       0.98      1.00      0.99       500
          es       1.00      0.99      1.00       500
          fr       1.00      1.00      1.00       500
          hi       1.00      0.95      0.98       500
          it       0.99      1.00      1.00       500
          ja       1.00      0.99      1.00       500
          nl       0.99      1.00      0.99       500
          pl       1.00      1.00      1.00       500
          pt       1.00      1.00      1.00       500
          ru       1.00      1.00      1.00       500
          sw       0.95      1.00      0.97       500
          th       1.00      1.00      1.00       500
          tr       0.99      1.00      1.00  

## 9. Reusable inference function

`detect_language(text)` is the function the rest of the pipeline (and the Flask app) will call.
It returns a dict with the predicted ISO language code and the model's confidence.


In [ ]:
def detect_language(text: str) -> dict:
    """Detect the language of a piece of text.

    Args:
        text: raw customer message.

    Returns:
        {"language": <iso_code str>, "confidence": <float 0-1>}
    """
    cleaned = basic_clean(text)
    if not cleaned:
        return {"language": "unknown", "confidence": 0.0}

    vec = vectorizer.transform([cleaned])
    proba = clf.predict_proba(vec)[0]
    pred_idx = int(np.argmax(proba))
    lang_code = label_encoder.inverse_transform([pred_idx])[0]
    confidence = float(proba[pred_idx])
    return {"language": lang_code, "confidence": round(confidence, 4)}


# Quick smoke test
for sample in [
    "Hello, I would like to track my order please.",
    "مرحبا، أريد معرفة حالة طلبي من فضلك",
    "Bonjour, je voudrais suivre ma commande.",
    "Hola, quiero saber el estado de mi pedido.",
]:
    print(sample, "->", detect_language(sample))


Hello, I would like to track my order please. -> {'language': 'en', 'confidence': 0.9419}
مرحبا، أريد معرفة حالة طلبي من فضلك -> {'language': 'ar', 'confidence': 0.9416}
Bonjour, je voudrais suivre ma commande. -> {'language': 'fr', 'confidence': 0.9133}
Hola, quiero saber el estado de mi pedido. -> {'language': 'es', 'confidence': 0.8782}


## 10. Mount Google Drive and set up the project folder structure

All 4 notebooks share one Drive folder, `RAG_chatbot_project/`, organized into one subfolder
per module so artifacts never collide and `app.py` can load each module from a predictable
path:

```
RAG_chatbot_project/
├── language_detection/      <- this notebook saves here
│   ├── language_tfidf_vectorizer.joblib
│   ├── language_classifier.joblib
│   └── language_label_encoder.joblib
├── sentiment_classifier/    <- Notebook 2 saves here
├── intent_classifier/       <- Notebook 3 saves here
└── rag_pipeline/            <- Notebook 4 saves here
```

This cell mounts Drive and creates the full structure (safe to re-run — `exist_ok=True` means
it won't error or wipe anything if the folders already exist).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/RAG_chatbot_project"
LANGUAGE_DIR = os.path.join(BASE_DIR, "language_detection")
SENTIMENT_DIR = os.path.join(BASE_DIR, "sentiment_classifier")
INTENT_DIR = os.path.join(BASE_DIR, "intent_classifier")
RAG_DIR = os.path.join(BASE_DIR, "rag_pipeline")

for d in [LANGUAGE_DIR, SENTIMENT_DIR, INTENT_DIR, RAG_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted. Project folder structure ready under:", BASE_DIR)


Mounted at /content/drive
Drive mounted. Project folder structure ready under: /content/drive/MyDrive/RAG_chatbot_project


## 11. Save artifacts

In [ ]:
joblib.dump(vectorizer, os.path.join(LANGUAGE_DIR, "language_tfidf_vectorizer.joblib"))
joblib.dump(clf, os.path.join(LANGUAGE_DIR, "language_classifier.joblib"))
joblib.dump(label_encoder, os.path.join(LANGUAGE_DIR, "language_label_encoder.joblib"))

print("Saved language detection artifacts to:", LANGUAGE_DIR)
print(f"Final test accuracy: {test_acc:.4f}")


Saved language detection artifacts to: /content/drive/MyDrive/RAG_chatbot_project/language_detection
Final test accuracy: 0.9941
